#Phase 4: Downside Risk

## 1. Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
data_path = project_root / "data" / "processed" / "daily_returns.csv"

daily_returns = pd.read_csv(
    data_path,
    index_col=0,
    parse_dates=True
)

portfolio_assets = ["AAPL", "JPM", "JNJ", "EFA", "LQD", "DBC"]
benchmark = "SPY"

expected_columns = portfolio_assets + [benchmark]

assert daily_returns.columns.tolist() == expected_columns, (
    f"Unexpected columns: {daily_returns.columns.tolist()}"
)

print("Dataset shape:", daily_returns.shape)
display(daily_returns.head())

Dataset shape: (2479, 7)


,AAPL,JPM,JNJ,EFA,LQD,DBC,SPY
Date,,,,,,,
2017-01-04,-0.001119,0.001845,-0.001640,0.011362,0.003411,0.007668,0.005949
2017-01-05,0.005085,-0.009205,0.010463,0.008510,0.005353,0.005073,-0.000795
2017-01-06,0.011148,0.000116,-0.004792,-0.003375,-0.005240,-0.001262,0.003578
2017-01-09,0.009160,0.000697,-0.000172,-0.001524,0.001954,-0.015161,-0.003301
2017-01-10,0.001009,0.002901,-0.001032,0.000170,-0.000594,-0.004490,0.000000


## 2. Load daily returns

In [3]:
# Define equal weights for the six portfolio holdings
weights = pd.Series(
    1 / len(portfolio_assets),
    index=portfolio_assets,
    name="Portfolio Weight"
)

# Validate portfolio weights
assert len(weights) == 6
assert np.isclose(weights.sum(), 1.0)
assert weights.index.tolist() == portfolio_assets

# Exclude SPY from portfolio return calculation
asset_returns = daily_returns[portfolio_assets]

# Calculate daily constant-weight portfolio returns
portfolio_returns = asset_returns.dot(weights)

print("Portfolio weights:")
display(weights)

print("\nPortfolio returns:")
display(portfolio_returns.head())

Portfolio weights:


AAPL    0.166667
JPM     0.166667
JNJ     0.166667
EFA     0.166667
LQD     0.166667
DBC     0.166667
Name: Portfolio Weight, dtype: float64


Portfolio returns:


Date
2017-01-04    0.003588
2017-01-05    0.004213
2017-01-06   -0.000568
2017-01-09   -0.000841
2017-01-10   -0.000340
dtype: float64

In [4]:
portfolio_value = (1 + portfolio_returns).cumprod()

portfolio_value.head()

Date
2017-01-04    1.003588
2017-01-05    1.007816
2017-01-06    1.007244
2017-01-09    1.006397
2017-01-10    1.006055
dtype: float64

In [5]:
running_max = portfolio_value.cummax()

running_max.head()

Date
2017-01-04    1.003588
2017-01-05    1.007816
2017-01-06    1.007816
2017-01-09    1.007816
2017-01-10    1.007816
dtype: float64

## 3. Maximum drawdown

In [6]:
drawdown = (portfolio_value - running_max) / running_max

drawdown.head()

Date
2017-01-04    0.000000
2017-01-05    0.000000
2017-01-06   -0.000568
2017-01-09   -0.001408
2017-01-10   -0.001747
dtype: float64

In [7]:
max_drawdown = drawdown.min()

print(f"Maximum Drawdown: {max_drawdown:.2%}")

Maximum Drawdown: -28.52%


In [10]:
# Calculate SPY drawdown
spy_returns = daily_returns[benchmark]

spy_value = (1 + spy_returns).cumprod()
spy_running_max = spy_value.cummax()

spy_drawdown = (
    (spy_value - spy_running_max) / spy_running_max
)

spy_max_drawdown = spy_drawdown.min()

print(f"SPY Maximum Drawdown: {spy_max_drawdown:.2%}")

SPY Maximum Drawdown: -33.72%


## 4. Value at Risk(VaR)

In [8]:
var_95 = portfolio_returns.quantile(0.05)

print(f"95% Daily VaR: {var_95:.2%}")

95% Daily VaR: -1.19%


In [9]:
spy_var_95 = daily_returns[benchmark].quantile(0.05)

print("Portfolio 95% Daily VaR:", round(var_95 * 100, 2), "%")
print("SPY 95% Daily VaR:", round(spy_var_95 * 100, 2), "%")

Portfolio 95% Daily VaR: -1.19 %
SPY 95% Daily VaR: -1.69 %


In [11]:
# Historical VaR exceedance analysis

portfolio_exceedances = (portfolio_returns < var_95).sum()
spy_exceedances = (spy_returns < spy_var_95).sum()

portfolio_exceedance_rate = (
    portfolio_exceedances / len(portfolio_returns)
)

spy_exceedance_rate = (
    spy_exceedances / len(spy_returns)
)

print("Portfolio VaR Exceedances:", portfolio_exceedances)
print(f"Portfolio Exceedance Rate: {portfolio_exceedance_rate:.2%}")

print("SPY VaR Exceedances:", spy_exceedances)
print(f"SPY Exceedance Rate: {spy_exceedance_rate:.2%}")

Portfolio VaR Exceedances: 124
Portfolio Exceedance Rate: 5.00%
SPY VaR Exceedances: 124
SPY Exceedance Rate: 5.00%
